Импортируем необходимые инструменты для работы программы:
pandas - для удобной работы с табличными данными (чтение и сохранение CSV-файлов)
re - для поиска и удаления текстового мусора с помощью гибких шаблонов
os - для работы с файловой системой (проверка существования файлов)


In [ ]:
import pandas as pd
import re
import os



=== НАСТРОЙКИ ПУТЕЙ ===
Здесь указываются пути к файлу, откуда берутся сырые данные, и файлу, куда сохранится результат.
Укажи точное имя твоего файла, если оно отличается от literaguru_ml_ready_1599.csv


In [ ]:
INPUT_FILE = r"C:\Users\user\Desktop\Работа\Почти финал\literaguru_ml_ready_1599.csv"
OUTPUT_FILE = r"C:\Users\user\Desktop\Работа\Почти финал\literaguru_elite_clean.csv"



Словарь для замены латиницы на кириллицу (хомоглифы)
Некоторые тексты могут содержать английские буквы "a, c, e..." вместо русских для обхода антиплагиата.
Эта настройка автоматически переводит их обратно в нормальный русский текст.


In [ ]:
HOMOGLYPHS = str.maketrans('aceopxyABCEHKMOPTX', 'асеорхуАВСЕНКМОРТХ')


# Блок 1: Функция для глубокой очистки каждого текста от мусора
def deep_clean_text(text):
    # Если вместо текста попалась пустота или ошибка — возвращаем пустую строку
    if not isinstance(text, str):
        return ""

    # 1. Убиваем латиницу (замена на кириллицу для защиты от искусственной уникальности)
    text = text.translate(HOMOGLYPHS)



2. Вырезаем даты (например, "24.04.2020" или "Обновлено 28.04.2020")
Удаляем следы публикации, чтобы текст выглядел чистым


In [ ]:
    text = re.sub(r'\d{2}\.\d{2}\.\d{4}(?:\s*Обновлено\s*\d{2}\.\d{2}\.\d{4})?', '', text, flags=re.IGNORECASE)



3. Вырезаем авторов и плашки (Автор: Guru, Иван Цыганок, Опубликовано)
Ищем в начале текста весь этот мусор и сносим его


In [ ]:
    text = re.sub(r'^(?:Автор:\s*.*?|Guru\s*|Иван Цыганок\s*|Опубликовано\s*|\s*·\s*)+', '', text, flags=re.IGNORECASE)

    # Добиваем имена авторов, если они застряли где-то еще в середине или конце текста
    text = re.sub(r'Автор\s*:.*?(?=\s|$)', '', text, flags=re.IGNORECASE)

    # 4. Чистим технические счетчики слов вроде "(500 слов)"
    text = re.sub(r'\(\s*\d+\s+слов[а-я]*\s*\)', '', text, flags=re.IGNORECASE)

    # 5. Косметика: убираем точки-буллиты, неразрывные системные пробелы и лишние пустые места
    text = text.replace('·', '').replace('\xa0', ' ')

    # Заменяем несколько подряд идущих пробелов на один аккуратный
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Блок 2: Функция для выявления бракованных текстов (ошибки при скачивании с сайта)
def is_broken_parsing(text):


Проверяет текст на наличие признаков битого парсинга (буквицы)


In [ ]:
    # Если текст начинается со строчной (маленькой) русской буквы - он битый и обрезанный
    if re.match(r'^[а-яё]', text):
        return True



Если слишком много предложений начинаются с маленькой буквы (оторванные абзацы)
Ищем точку/знак вопроса, пробел и сразу маленькую букву


In [ ]:
    broken_sentences = re.findall(r'[.!?]\s+[а-яё]', text)
    if len(broken_sentences) > 2:  # Если таких косяков больше 2 на весь текст - отправляем его в мусор
        return True

    # Если текст прошел все проверки — он хороший
    return False


# Блок 3: Главная функция, которая по очереди запускает все процессы
def main():
    # Проверяем, существует ли исходный файл по указанному пути
    if not os.path.exists(INPUT_FILE):
        print(f"[ОШИБКА] Файл не найден по пути:\n{INPUT_FILE}")
        print("Проверь, правильное ли имя файла указано в коде.")
        return

    # Шаг 1: Загружаем данные из файла в память программы
    print("[1/3] Читаем исходный файл...")
    df = pd.read_csv(INPUT_FILE, sep=';', encoding='utf-8-sig')
    initial_len = len(df)  # Запоминаем изначальное количество текстов для финальной статистики

    # Шаг 2: Применяем функцию очистки (deep_clean_text) к каждой строке с текстом
    print("[2/3] Запускаем глубокую очистку (чистим мусор и латиницу)...")
    df['Текст'] = df['Текст'].apply(deep_clean_text)

    # Шаг 3: Выкидываем плохие тексты и дубликаты
    print("[3/3] Фильтруем битый парсинг и дубликаты...")

    # Оставляем только те тексты, которые НЕ являются битыми и длина которых больше 300 символов
    mask = df['Текст'].apply(lambda x: not is_broken_parsing(x) and len(x) > 300)
    df = df[mask]



Создаем скрытую колонку, где текст приведен к нижнему регистру без знаков препинания.
Это нужно для защиты от смысловых дублей (чтобы удалить одинаковые тексты, даже если у них разная пунктуация)


In [ ]:
    df['norm'] = df['Текст'].apply(lambda x: re.sub(r'[^а-яё0-9]', '', str(x).lower()))

    # Удаляем дубликаты на основе этой скрытой колонки
    df = df.drop_duplicates(subset=['norm'])

    # Убираем техническую колонку, так как она больше не нужна
    df = df.drop(columns=['norm'])

    # Шаг 4: Сохраняем итоговый, идеально чистый файл
    df.to_csv(OUTPUT_FILE, index=False, sep=';', quoting=1, encoding='utf-8-sig')

    # Шаг 5: Выводим финальную статистику на экран
    final_len = len(df)
    print("\n=== ГОТОВО ===")
    print(f"Было текстов: {initial_len}")
    print(f"Осталось идеальных: {final_len}")
    print(f"Уничтожено мусорных/битых: {initial_len - final_len}")
    print(f"\nФайл сохранен: {OUTPUT_FILE}")


# Это стандартная техническая команда для запуска программы
if __name__ == "__main__":
    main()
